# Suite SVA — The SystemVerilog Assertions dialect

SystemVerilog Assertions' sequences and properties (IEEE 1800 16.7 to 16.12) are a dialect of the SystemVerilog family,
`Expressions.SystemVerilog.Assertions`: mbse-programs' kinds, in its shapes, each a branch of its `Term`, which lists
every branch of `Expressions`' `Term` besides its own, since a position that holds a property holds a sequence, a
property or an expression. A call's arguments hold it, so a tree mixes its kinds with the family's; it is validated and
printed from any of the family's holders, its editions are data, and evaluation refuses each of its kinds by name.

In [ ]:
from mbse.Expressions import Framework
from mbse.Expressions.IEEE1164 import Expressions as IEEE1164
from mbse.Expressions.IEEE754 import Expressions as IEEE754
from mbse.Expressions.ISOIEC11404 import Expressions as ISOIEC11404
from mbse.Expressions.Unicode import Expressions as Unicode
from mbse.Expressions.Patterns.Expressions import Expressions as PE
from mbse.Expressions.SystemVerilog import Assertions as SVAssertions, Constraints as SVConstraints, \
    Expressions as SystemVerilog, Patterns as SVPatterns, Preprocessor as SVPreprocessor, Types as SVTypes
from mbse.Expressions.SystemVerilog.Assertions import Expressions as SVA, Text as SVAText
from mbse.Expressions.SystemVerilog.Expressions import Evaluators as SVV, Expressions as SV, Text as SVText
from mbse.Expressions.SystemVerilog.Types import Expressions as SVT
from mbse.Expressions.Framework import Domains as D, Standards as FS, Symbolics as SY, Terms as F
from mbse.Schemas.Framework import Builders, JSON, Validators, YAML
from support import raises

store = Builders.Store()  # the SystemVerilog family's six dialects, after the standard dialects they use
for install in [Framework.install, ISOIEC11404.install, IEEE754.install, IEEE1164.install, Unicode.install,
                SystemVerilog.install, SVTypes.install, SVPatterns.install, SVAssertions.install, SVConstraints.install,
                SVPreprocessor.install]:
    store.transaction(install)
for define in [SystemVerilog.define, SVAssertions.define]:  # SystemVerilog's types, and the editions Assertions covers
    store.transaction(define)
Term = store.registered("Expressions.Framework.Term")
UNION, EXPRESSIONS = store.registered(SVA.DIALECT.union), store.registered(SV.DIALECT.union)
KINDS = {b.name: b.schema for b in UNION.branches}  # its Term's branches' schemas, by tag
BOUND = {"a", "b", "c", "d", "clk", "rst", "en", "n", "x", "f"}  # the names its samples refer to


def held(spec, holder="Assertions"):
    """The term value `spec` writes, the root of a holder of the family's dialect `holder` written in a transaction."""
    with store.transaction():
        return F.term(getattr(store.Expressions.SystemVerilog, holder).Expression(lambda e: e.root(spec)))


def validate(spec, bound=BOUND):
    """The problems of the term `spec` writes, held by Assertions' holder, with `bound` bound."""
    return SVA.DIALECT.validate(held(spec), bound)


def text(spec):
    """The term `spec` writes, held by Assertions' holder, as SystemVerilog source."""
    return SVText.ToText(held(spec))


def round_trips(value):
    """Whether a term of its Term reads back the same from JSON and from YAML."""
    return all(F.same(decode(UNION, encode(UNION, value)), value) for encode, decode in [
        (JSON.ToJSON(store), JSON.FromJSON(store)), (YAML.ToYAML(store), YAML.FromYAML(store))])


spelled = lambda text: lambda i: i.spelling(text)  # noqa: E731: an identifier, of its spelling
name = lambda text: lambda u: u.nameexpression(lambda n: n.name(lambda i: i.identifier(spelled(text))))  # noqa: E731
a, b, c, d, clk, rst, en, n, x = (name(t) for t in ("a", "b", "c", "d", "clk", "rst", "en", "n", "x"))
one, two, three = SV.literal(1), SV.literal(2), SV.literal(3)
dollar = lambda u: u.dollarexpression(lambda r: r)  # noqa: E731: `$`
cycles = lambda low, high: lambda u: u.cyclerange(lambda r: r.low(low).high(high))  # noqa: E731: `low:high`
event = lambda edge, signal: lambda e: e.edge(edge).expression(signal)  # noqa: E731: `edge signal`
posedge = lambda k: k.events([event("posedge", clk)])  # noqa: E731: a clock, `@(posedge clk)`
seq = lambda left, op, right: lambda u: u.binarysequence(lambda s: s.left(left).operator(op).right(right))  # noqa: E731
prop = lambda left, op, right: lambda u: u.binaryproperty(lambda s: s.left(left).operator(op).right(right))  # noqa: E731
implies = lambda ante, op, cons: lambda u: u.implicationproperty(  # noqa: E731
    lambda s: s.antecedent(ante).operator(op).consequent(cons))
unary = lambda op, operand, range_=None: lambda u: u.unaryproperty(  # noqa: E731
    lambda s: (s.operator(op).range(range_) if range_ is not None else s.operator(op)).operand(operand))
delay = lambda first, *steps: lambda u: u.delaysequence(lambda s: (s.first(first) if first is not None else s).steps(  # noqa: E731
    [lambda t, step=step: t.delay(step[0]).sequence(step[1]) for step in steps]))
repeat = lambda sequence, op, count: lambda u: u.repetitionsequence(  # noqa: E731
    lambda s: s.sequence(sequence).operator(op).count(count))

<a id="SVA-01"></a>
## SVA-01 · Its union and holder: its kinds after `Expressions`' branches, in mbse-programs' shapes, the positions that hold its `Term`, its holder validated, linked by `HasRoot` and round-tripped

In [ ]:
# Its kinds: mbse-programs' sequences and properties, a property spec and a cycle range, after Expressions' branches and
# invalid syntax; the Preprocessor's macro usage after them, and the Constraints' distexpression once its install adds it.
assert SVA.TAGS == (*SVA.SEQUENCES, *SVA.PROPERTIES, "propertyspec", "cyclerange") and len(SVA.TAGS) == 17
assert SVA.SEQUENCES == ("delaysequence", "repetitionsequence", "binarysequence", "parenthesizedsequence",
                         "firstmatchsequence", "clockedsequence")
assert SVA.PROPERTIES == ("implicationproperty", "binaryproperty", "unaryproperty", "strengthproperty", "abortproperty",
                          "conditionalproperty", "caseproperty", "parenthesizedproperty", "clockedproperty")
assert [b.name for b in UNION.branches if b.name != "distexpression"] == [*SV.TAGS, "error", "missing", *SVA.TAGS, "macrousage"]
assert list(SVA.DIALECT.kinds()) == [*SVA.TAGS, "error", "missing"] and SVA.HELPERS == (
    "DelayStep", "PropertyCaseItem", "EventControl", "EventExpression")
assert SVA.DIALECT.name() == "SystemVerilog Assertions" and SVA.DIALECT.tag == "systemverilog_assertions"
assert {b.name: b.schema for b in Term.branches}["systemverilog_assertions"] is UNION  # the framework's Term's branch
assert all(KINDS[tag] is {b.name: b.schema for b in EXPRESSIONS.branches}[tag] for tag in SV.TAGS)  # one schema each
# Each kind a value object, its properties in mbse-programs' order; every operand of a sequence or a property, and what
# the standard writes `expression_or_dist`, holds its Term; a match item, a cycle range's bound and an event Expressions'.
PROPERTIES = {
    "delaysequence": ["first", "steps"], "repetitionsequence": ["sequence", "operator", "count"],
    "binarysequence": ["left", "operator", "right"], "parenthesizedsequence": ["sequence", "items"],
    "firstmatchsequence": ["sequence", "items"], "clockedsequence": ["clock", "sequence"],
    "implicationproperty": ["antecedent", "operator", "consequent"], "binaryproperty": ["left", "operator", "right"],
    "unaryproperty": ["operator", "range", "operand"], "strengthproperty": ["keyword", "sequence"],
    "abortproperty": ["keyword", "condition", "operand"], "conditionalproperty": ["condition", "consequence", "alternative"],
    "caseproperty": ["expression", "items"], "parenthesizedproperty": ["property"], "clockedproperty": ["clock", "property"],
    "propertyspec": ["clock", "disable", "property"], "cyclerange": ["low", "high"],
}
assert all(list(KINDS[tag].properties) == PROPERTIES[tag] and not KINDS[tag].ref for tag in SVA.TAGS)
assert all(KINDS[tag].name == f"Expressions.SystemVerilog.Assertions.{KINDS[tag].name.rpartition('.')[2]}"
           and KINDS[tag].name.rpartition(".")[2].lower() == tag for tag in SVA.TAGS)
helper = lambda name: store.registered(f"Expressions.SystemVerilog.Assertions.{name}")  # noqa: E731
assert [list(helper(h).properties) for h in SVA.HELPERS] == [
    ["delay", "sequence"], ["expressions", "body"], ["events"], ["edge", "expression", "condition"]]
for tag, slot in [("binarysequence", "left"), ("delaysequence", "first"), ("unaryproperty", "range"),
                  ("conditionalproperty", "condition"), ("abortproperty", "condition"), ("caseproperty", "expression"),
                  ("propertyspec", "disable"), ("propertyspec", "property")]:
    assert KINDS[tag].properties[slot].schema is UNION, (tag, slot)
assert KINDS["cyclerange"].properties["low"].schema is EXPRESSIONS and KINDS["parenthesizedsequence"].properties[
    "items"].schema.element is EXPRESSIONS and helper("EventExpression").properties["condition"].schema is EXPRESSIONS
assert helper("DelayStep").properties["delay"].schema is UNION and helper("PropertyCaseItem").properties[
    "expressions"].schema.element is UNION and KINDS["clockedsequence"].properties["clock"].schema is helper("EventControl")
# How the framework reads them: a delay's steps, pairs of terms, its variadic; a clock and a case's items helpers.
assert F._parts(KINDS["delaysequence"]) == (("first",), "steps", ()) and F._parts(KINDS["parenthesizedsequence"]) == (
    ("sequence",), "items", ())
assert F._parts(KINDS["clockedsequence"]) == (("sequence",), None, ("clock",)) and F._parts(KINDS["caseproperty"]) == (
    ("expression",), None, ("items",)) and F._parts(KINDS["propertyspec"]) == (("disable", "property"), None, ("clock",))
assert all(SVA.DIALECT.kinds()[tag].role == F.APPLICATION for tag in SVA.TAGS)
# Its holder (Q2, "(a)") refines the framework's Expression, its root narrowed to its Term: it validates, holds a term of
# its dialect, round-trips through JSON and YAML, and HasRoot links it, as any expression.
Expression = store.registered("Expressions.Framework.Expression")
Holder = store.registered("Expressions.SystemVerilog.Assertions.Expression")
assert Holder.ref and Holder.refines is Expression and F.is_expression(Holder) and Holder.properties["root"].schema is UNION
with store.transaction():
    spec = store.Expressions.SystemVerilog.Assertions.Expression(lambda e: e.root(lambda u: u.propertyspec(
        lambda s: s.clock(posedge).disable(rst).property(implies(a, "|=>", b)))))
    linked = store.Schemas.OfExpression(lambda s: s.name("sva.Handshake").expression(
        lambda h: h.insert(lambda e: e.expression(spec))))
assert Validators.Validate(store)(Holder, spec) == [] and SVA.DIALECT.validate(spec, BOUND) == []
assert F.dialect_of(F.term(spec)) == SVA.DIALECT.union and round_trips(F.term(spec))
assert F.same(JSON.FromJSON(store)(Holder, JSON.ToJSON(store)(Holder, spec)).root, spec.root)
assert [F.same(entry.expression, spec) for entry in Builders.OfExpression.Builder(linked, False, store).expression] == [True]
assert SVText.ToText(spec) == SVAText.ToText(spec) == "@(posedge clk) disable iff (rst) a |=> b"
with raises(Exception):  # the schema refuses a kind its root's union lacks: a Types term
    with store.transaction():
        store.Expressions.SystemVerilog.Assertions.Expression(lambda e: e.root(lambda t: t.integeratomtype(lambda i: i.keyword("int"))))

<a id="SVA-02"></a>
## SVA-02 · Sequences: each kind written through the builders and validated, its checks with mbse-programs' messages, and no kind binding a name

In [ ]:
# Each sequence (16.7 to 16.10), written through the builders: validated, its names free (none of its kinds binds one),
# and its checks, with mbse-programs' choices and messages.
match_item = SV.binary("+", x, one)  # a match item, an expression
SEQUENCES = [
    delay(a, (one, b), (cycles(one, dollar), c)),  # a ##1 b ##[1:$] c
    delay(None, (two, a)),  # ##2 a: a sequence that starts with a delay
    repeat(a, "*", two), repeat(b, "->", cycles(one, three)), repeat(c, "=", n),
    seq(a, "and", b), seq(a, "or", b), seq(a, "intersect", b), seq(a, "within", b), seq(en, "throughout", b),
    lambda u: u.parenthesizedsequence(lambda s: s.sequence(a).items([match_item])),
    lambda u: u.firstmatchsequence(lambda s: s.sequence(seq(a, "or", b))),
    lambda u: u.clockedsequence(lambda s: s.clock(posedge).sequence(a)),
]
assert [validate(spec) for spec in SEQUENCES] == [[]] * len(SEQUENCES)
assert [F.kind_of(held(spec))[0] for spec in SEQUENCES] == [
    "delaysequence", "delaysequence", *["repetitionsequence"] * 3, *["binarysequence"] * 5, "parenthesizedsequence",
    "firstmatchsequence", "clockedsequence"]
assert SY.free(held(SEQUENCES[0]), SVA.DIALECT) == {"a", "b", "c"} and SY.free(held(SEQUENCES[4]), SVA.DIALECT) == {"c", "n"}
assert SY.free(held(SEQUENCES[10]), SVA.DIALECT) == {"a", "x"} and SY.free(held(SEQUENCES[12]), SVA.DIALECT) == {"clk", "a"}
assert not any(F.is_binder(KINDS[tag]) for tag in SVA.TAGS)  # no kind of it binds a name: no `Let` refined
assert validate(SEQUENCES[0], {"a", "b"}) == ["sequence 1: nameexpression 'c' is not bound"]  # a step's members, by index
assert SVA.DIALECT.infer(held(SEQUENCES[5]), {"a": D.Anything, "b": D.Anything}) is D.Anything  # nothing it evaluates
for spec, problems in [
    (repeat(a, "+", two), ["RepetitionSequence.operator cannot be '+'"]),
    (seq(a, "xor", b), ["BinarySequence.operator cannot be 'xor'"]),
    (lambda u: u.binarysequence(lambda s: s.left(a).right(b)), ["a binarysequence needs an operator"]),
    (lambda u: u.binarysequence(lambda s: s.left(a).operator("and")), ["a binarysequence needs a right"]),
    (lambda u: u.delaysequence(lambda s: s.steps([lambda t: t.delay(one)])), ["a delaysequence needs a sequence 0"]),
    (lambda u: u.clockedsequence(lambda s: s.sequence(a)), ["a clockedsequence needs a clock"]),
    (lambda u: u.clockedsequence(lambda s: s.clock(lambda k: k.events([event("rise", clk)])).sequence(a)),
     ["EventExpression.edge cannot be 'rise'"]),  # a helper's choice, read within the clock
    (lambda u: u.clockedsequence(lambda s: s.clock(lambda k: k.events([lambda e: e.edge("posedge")])).sequence(a)),
     ["a clockedsequence needs a clock events 0 expression"]),
    (lambda u: u.clockedsequence(lambda s: s.clock(lambda k: k.events([event("edge", name("q"))])).sequence(a)),
     ["clock events 0 expression: nameexpression 'q' is not bound"]),
    (lambda u: u.clockedsequence(lambda s: s.clock(lambda k: k.events([lambda e: e.expression(clk).condition(en)])).sequence(a)),
     []),  # an edge and a condition may be absent
    (lambda u: u.parenthesizedsequence(lambda s: s.sequence(a).items([name("q")])), ["argument 0: nameexpression 'q' is not bound"]),
]:
    assert validate(spec) == problems, (problems, validate(spec))

<a id="SVA-03"></a>
## SVA-03 · Properties, a property spec and a cycle range: each kind validated, its checks, and the names it needs

In [ ]:
# Each property (16.12), a property spec and a cycle range: validated, its names free, and its checks.
case_items = lambda i, j: [lambda t: t.expressions([one, two]).body(i), lambda t: t.body(j)]  # noqa: E731: `1, 2: i; default: j;`
PROPERTIES = [
    implies(a, "|->", b), implies(a, "|=>", b), implies(a, "#-#", b), implies(a, "#=#", b),
    *(prop(a, op, b) for op in ("and", "or", "iff", "implies", "until", "s_until", "until_with", "s_until_with")),
    *(unary(op, a) for op in ("not", "nexttime", "s_nexttime", "always", "s_always", "eventually", "s_eventually")),
    unary("nexttime", a, two), unary("always", a, cycles(one, three)),
    *(lambda u, k=k: u.strengthproperty(lambda s: s.keyword(k).sequence(seq(a, "or", b))) for k in ("strong", "weak")),
    *(lambda u, k=k: u.abortproperty(lambda s: s.keyword(k).condition(rst).operand(a))
      for k in ("accept_on", "reject_on", "sync_accept_on", "sync_reject_on")),
    lambda u: u.conditionalproperty(lambda s: s.condition(en).consequence(a)),
    lambda u: u.conditionalproperty(lambda s: s.condition(en).consequence(a).alternative(b)),
    lambda u: u.caseproperty(lambda s: s.expression(n).items(case_items(a, b))),
    lambda u: u.parenthesizedproperty(lambda s: s.property(implies(a, "|->", b))),
    lambda u: u.clockedproperty(lambda s: s.clock(posedge).property(a)),
    lambda u: u.propertyspec(lambda s: s.clock(posedge).disable(rst).property(a)),
    lambda u: u.propertyspec(lambda s: s.property(a)),  # with neither clock nor reset
    cycles(one, dollar),
]
assert [validate(spec) for spec in PROPERTIES] == [[]] * len(PROPERTIES)
assert {F.kind_of(held(spec))[0] for spec in PROPERTIES} == {*SVA.PROPERTIES, "propertyspec", "cyclerange"}
assert SY.free(held(PROPERTIES[-6]), SVA.DIALECT) == {"n", "a", "b"} and SY.free(held(PROPERTIES[-3]), SVA.DIALECT) == {
    "clk", "rst", "a"}
for spec, problems in [
    (implies(a, "->", b), ["ImplicationProperty.operator cannot be '->'"]),
    (prop(a, "xor", b), ["BinaryProperty.operator cannot be 'xor'"]),
    (unary("never", a), ["UnaryProperty.operator cannot be 'never'"]),
    (lambda u: u.unaryproperty(lambda s: s.operand(a)), ["an unaryproperty needs an operator"]),
    (lambda u: u.strengthproperty(lambda s: s.keyword("firm").sequence(a)), ["StrengthProperty.keyword cannot be 'firm'"]),
    (lambda u: u.abortproperty(lambda s: s.keyword("abort_on").condition(rst).operand(a)),
     ["AbortProperty.keyword cannot be 'abort_on'"]),
    (lambda u: u.abortproperty(lambda s: s.condition(rst).operand(a)), ["an abortproperty needs a keyword"]),
    (lambda u: u.conditionalproperty(lambda s: s.consequence(a)), ["a conditionalproperty needs a condition"]),
    (lambda u: u.caseproperty(lambda s: s.expression(n).items([lambda t: t.expressions([one])])),
     ["a caseproperty needs an items 0 body"]),
    (lambda u: u.caseproperty(lambda s: s.expression(n).items([lambda t: t.expressions([name("q")]).body(a)])),
     ["items 0 argument 0: nameexpression 'q' is not bound"]),
    (lambda u: u.clockedproperty(lambda s: s.property(a)), ["a clockedproperty needs a clock"]),
    (lambda u: u.propertyspec(lambda s: s.disable(name("q"))), ["disable: nameexpression 'q' is not bound",
                                                                "a propertyspec needs a property"]),
    (unary("always", a, lambda u: u.cyclerange(lambda r: r.low(one))), ["range: a cyclerange needs a high"]),
]:
    assert validate(spec) == problems, (problems, validate(spec))

<a id="SVA-04"></a>
## SVA-04 · A tree mixing its kinds with the family's, validated and printed from its holder and from `Expressions`', read by each dialect of the family and refused by another

In [ ]:
# A tree mixing its kinds with the family's: an expression, a cast to a Types term and a macro usage as a sequence's
# operands, validated and printed from its holder; and a sequence as a call's argument, which holds its Term, validated
# and printed from Expressions' holder, read through every dialect of the family; another dialect refuses it.
int_ = lambda t: t.integeratomtype(lambda i: i.keyword("int"))  # noqa: E731
macro = lambda u: u.macrousage(lambda m: m.name(spelled("READY")))  # noqa: E731
mixed = lambda u: u.propertyspec(lambda s: s.clock(lambda k: k.events([event("posedge", clk), lambda e: e.edge("negedge").expression(
    rst).condition(en)])).disable(SV.binary("||", rst, SV.unary("!", en))).property(implies(
        delay(SV.binary("==", SV.cast(int_, a), one), (n, macro)), "|=>", unary("always", seq(b, "or", SV.select(c, d))))))
spec = held(mixed)
assert SVA.DIALECT.validate(spec, BOUND) == [] and SY.free(spec, SVA.DIALECT) == {"clk", "rst", "en", "a", "n", "b", "c", "d"}
assert SVA.DIALECT.validate(spec, BOUND - {"d"}) == [
    "property: consequent: operand: right: index: nameexpression 'd' is not bound"]
assert SVText.ToText(spec) == ("@(posedge clk or negedge rst iff en) disable iff (rst || !en) "
                               "int'(a) == 1 ##n `READY |=> always b or c[d]")
cast = F.form(F.form(F.form(F.form(F.form(F.term(spec)).arguments[1]).arguments[0]).arguments[0]).arguments[0]).arguments[0]
assert F.dialect_of(cast) == SVT.DIALECT.union and SVT.DIALECT.validate(cast) == []  # a Types term, within Expressions'
called = held(SV.call("f", delay(a, (one, b)), x), "Expressions")  # f(a ##1 b, x): a sequence is an argument
assert SV.DIALECT.validate(called, BOUND) == [] and SVText.ToText(called) == "f(a ##1 b, x)"
argument = F.form(called).arguments[1]
assert F.dialect_of(argument) == SVA.DIALECT.union and F.kind_of(argument)[0] == "delaysequence"
assert all(d.kind(argument)[2] is SVA.DIALECT.kinds()["delaysequence"] for d in (SV.DIALECT, *SV.DIALECT.reads()))
assert SV.DIALECT.validate(held(SV.call("f", delay(a, (one, name("q")))), "Expressions"), BOUND) == [
    "argument 0: sequence 0: nameexpression 'q' is not bound"]
assert PE.DIALECT.validate(argument) == ["a SystemVerilog Assertions delaysequence is not a Patterns term"]
with raises(ValueError, match="a SystemVerilog Assertions delaysequence is not a Patterns term"):
    PE.DIALECT.kind(argument)

<a id="SVA-05"></a>
## SVA-05 · Invalid syntax wherever its `Term` stands: reported, printed as written, and refused by evaluation

In [ ]:
# Invalid syntax (the framework's `error` and `missing`) stands wherever its Term does: as a root, an operand, a delay or
# a case item's body; reported as a problem, printed as written (a missing term as nothing), and refused by evaluation.
error = lambda written: lambda u: u.error(lambda e: e.text(written))  # noqa: E731
missing = lambda kind: lambda u: u.missing(lambda m: m.kind(kind))  # noqa: E731
assert validate(error("##")) == ["invalid syntax '##'"] and validate(missing("Sequence")) == ["missing Sequence"]
assert validate(seq(error("@@"), "and", missing("Sequence"))) == ["left: invalid syntax '@@'", "right: missing Sequence"]
assert validate(delay(a, (error("#1"), b))) == ["delay 0: invalid syntax '#1'"]
assert validate(lambda u: u.caseproperty(lambda s: s.expression(n).items([lambda t: t.body(missing("Property"))]))) == [
    "items 0 body: missing Property"]
assert text(seq(error("@@"), "and", missing("Sequence"))) == "@@ and "
assert text(delay(a, (error("#1"), b))) == "a ##(#1) b" and text(unary("always", error("p q"))) == "always p q"
assert text(implies(missing("Sequence"), "|->", b)) == " |-> b"
with raises(ValueError, match="cannot evaluate an invalid expression: invalid syntax '##'"):
    SVV.OfAny(held(error("##")), {})

<a id="SVA-06"></a>
## SVA-06 · Its editions as data: IEEE Std 1800-2005 to 1800-2023, each with its own kinds it has

In [ ]:
# Its editions are data (CAP-268), written by its define: IEEE 1800's five, each owned by its designation, with its formal
# reference's parts and its common name, and each with its own kinds it has, from mbse-programs' SINCE: a strength, an
# abort and a case property arrived in IEEE 1800-2009, every other kind in IEEE 1800-2005. Invalid syntax, the kinds of
# the dialects it reads and Verilog's editions (IEEE 1364) are none of its.
IEEE1800 = tuple(f"IEEE Std 1800-{year}" for year in (2005, 2009, 2012, 2017, 2023))
LABELS = tuple(f"SystemVerilog-{year}" for year in (2005, 2009, 2012, 2017, 2023))
singleton = store.singleton("Expressions.SystemVerilog.Assertions")
assert tuple(FS.standards(singleton)) == IEEE1800 and SVA.SINCE == {"strengthproperty": 4, "abortproperty": 4, "caseproperty": 4}
assert FS.tags(FS.find(singleton, "SystemVerilog-2009")) == ("IEEE Std 1800-2009", "SystemVerilog-2009")
assert FS.find(singleton, "IEEE Std 1800-2009") is FS.find(singleton, "SystemVerilog-2009") and FS.find(singleton, "Verilog-2005") is None
assert FS.properties(FS.find(singleton, "SystemVerilog-2023")) == {
    "bodies": ("org.ieee",), "number": "1800", "revision": ("year", 2023), "names": ("SystemVerilog-2023",)}
editions = {b.name: tuple(FS.tags(s)[-1] for s in FS.of(singleton, b.schema).values()) for b in UNION.branches}
assert editions == {tag: LABELS[SVA.SINCE.get(tag, 3) - 3:] if tag in SVA.TAGS else () for tag in editions}
assert [len(FS.kinds(FS.find(singleton, label))) for label in LABELS] == [14, 17, 17, 17, 17]
assert [FS.designation_of(e.standard) for e in Builders.OfObject.Builder(KINDS["caseproperty"], False, store).standards] == list(
    IEEE1800[1:])  # a kind's schema reads its editions from its side
assert store.transaction(SVAssertions.define)[0]["SystemVerilog_2005"] is FS.find(singleton, "SystemVerilog-2005")  # again: nothing new

<a id="SVA-07"></a>
## SVA-07 · Printing every kind as mbse-programs writes it: Table 16-3's precedence and associativity, delays and clocks

In [ ]:
# Printing every kind, as mbse-programs' printer writes it: by the precedence of sequence and property operators (Table
# 16-3), right-associative where the standard is, parenthesized only where needed; one that reaches as far right as it
# can (`always`, `if`, a clock, an abort) needs none where nothing follows it. A delay is written after `##` bare where it
# is an integer, a name, a parenthesized expression or a macro usage, else in parentheses.
clocked = lambda u: u.clockedsequence(lambda s: s.clock(posedge).sequence(b))  # noqa: E731: @(posedge clk) b
if_ = lambda c_, t, e=None: lambda u: u.conditionalproperty(  # noqa: E731
    lambda s: s.condition(c_).consequence(t).alternative(e) if e is not None else s.condition(c_).consequence(t))
abort = lambda u: u.abortproperty(lambda s: s.keyword("accept_on").condition(rst).operand(a))  # noqa: E731
for spec, written in [
    # sequences
    (delay(a, (one, b), (cycles(one, dollar), c)), "a ##1 b ##[1:$] c"), (delay(None, (two, a)), "##2 a"),
    (delay(a, (n, b)), "a ##n b"), (delay(a, (SV.binary("+", n, one), b)), "a ##(n + 1) b"),
    (delay(a, (lambda u: u.parenthesizedexpression(lambda p: p.expression(n)), b)), "a ##(n) b"),
    (delay(a, (lambda u: u.macrousage(lambda m: m.name(spelled("D"))), b)), "a ##`D b"),
    (delay(a, (SV.vector("01"), b)), "a ##2'b01 b"), (delay(a, (SV.literal(1.5), b)), "a ##(1.5) b"),
    (delay(a, (lambda u: u.constant(lambda k: k.fill("1")), b)), "a ##('1) b"),
    (delay(a, (one, seq(b, "or", c))), "a ##1 (b or c)"), (delay(seq(a, "or", b), (one, c)), "(a or b) ##1 c"),
    (repeat(a, "*", two), "a[*2]"), (repeat(b, "->", cycles(one, three)), "b[->1:3]"), (repeat(c, "=", n), "c[=n]"),
    (repeat(seq(a, "or", b), "*", two), "(a or b)[*2]"), (repeat(delay(a, (one, b)), "*", two), "(a ##1 b)[*2]"),
    (seq(seq(a, "and", b), "or", c), "a and b or c"), (seq(a, "and", seq(b, "or", c)), "a and (b or c)"),
    (seq(a, "throughout", seq(b, "throughout", c)), "a throughout b throughout c"),
    (seq(seq(a, "throughout", b), "throughout", c), "(a throughout b) throughout c"),
    (seq(a, "within", seq(b, "within", c)), "a within (b within c)"), (seq(a, "intersect", b), "a intersect b"),
    (seq(delay(a, (one, b)), "and", c), "a ##1 b and c"), (seq(SV.binary("&&", a, b), "and", c), "a && b and c"),
    (lambda u: u.parenthesizedsequence(lambda s: s.sequence(delay(a, (one, b))).items([SV.binary("+", x, one), n])),
     "(a ##1 b, x + 1, n)"),
    (lambda u: u.firstmatchsequence(lambda s: s.sequence(seq(a, "or", b))), "first_match(a or b)"),
    (clocked, "@(posedge clk) b"), (seq(clocked, "and", c), "(@(posedge clk) b) and c"),
    (seq(a, "and", clocked), "a and @(posedge clk) b"), (delay(a, (one, clocked)), "a ##1 @(posedge clk) b"),
    (delay(a, (one, clocked), (one, c)), "a ##1 (@(posedge clk) b) ##1 c"),
    (lambda u: u.clockedsequence(lambda s: s.clock(lambda k: k).sequence(a)), "@(*) a"),
    (lambda u: u.clockedsequence(lambda s: s.clock(lambda k: k.events([lambda e: e.edge("edge").expression(clk).condition(en),
                                                                      lambda e: e.expression(rst)])).sequence(a)),
     "@(edge clk iff en or rst) a"),
    # properties
    *((implies(a, op, b), f"a {op} b") for op in ("|->", "|=>", "#-#", "#=#")),
    (implies(a, "|->", implies(b, "|->", c)), "a |-> b |-> c"), (implies(implies(a, "|->", b), "|=>", c), "(a |-> b) |=> c"),
    (implies(delay(a, (one, b)), "|->", seq(c, "or", d)), "a ##1 b |-> c or d"),
    *((prop(a, op, b), f"a {op} b") for op in ("and", "or", "iff", "implies", "until", "s_until", "until_with", "s_until_with")),
    (prop(prop(a, "and", b), "or", c), "a and b or c"), (prop(a, "and", prop(b, "or", c)), "a and (b or c)"),
    (prop(a, "until", prop(b, "until", c)), "a until b until c"), (prop(prop(a, "iff", b), "iff", c), "(a iff b) iff c"),
    (prop(implies(a, "|->", b), "and", c), "(a |-> b) and c"),
    *((unary(op, a), f"{op} a") for op in ("not", "nexttime", "s_nexttime", "always", "s_always", "eventually", "s_eventually")),
    (unary("nexttime", a, two), "nexttime [2] a"), (unary("always", a, cycles(one, three)), "always [1:3] a"),
    (unary("not", prop(a, "and", b)), "not (a and b)"), (prop(unary("not", a), "and", b), "not a and b"),
    (unary("always", prop(a, "and", b)), "always a and b"), (prop(unary("always", a), "and", b), "(always a) and b"),
    (prop(a, "and", unary("always", b)), "a and always b"), (implies(a, "|->", unary("s_eventually", b)), "a |-> s_eventually b"),
    (lambda u: u.strengthproperty(lambda s: s.keyword("strong").sequence(seq(a, "or", b))), "strong(a or b)"),
    (lambda u: u.strengthproperty(lambda s: s.keyword("weak").sequence(a)), "weak(a)"),
    (abort, "accept_on (rst) a"), (prop(abort, "or", b), "(accept_on (rst) a) or b"), (prop(b, "or", abort), "b or accept_on (rst) a"),
    *((lambda u, k=k: u.abortproperty(lambda s: s.keyword(k).condition(rst).operand(a)), f"{k} (rst) a")
      for k in ("reject_on", "sync_accept_on", "sync_reject_on")),
    (if_(en, a), "if (en) a"), (if_(en, a, b), "if (en) a else b"), (if_(en, if_(rst, a), b), "if (en) (if (rst) a) else b"),
    (if_(en, a, if_(rst, b, c)), "if (en) a else if (rst) b else c"), (prop(if_(en, a), "and", b), "(if (en) a) and b"),
    (lambda u: u.caseproperty(lambda s: s.expression(n).items([lambda t: t.expressions([one, two]).body(a),
                                                               lambda t: t.body(implies(b, "|->", c))])),
     "case (n) 1, 2: a; default: b |-> c; endcase"),
    (lambda u: u.parenthesizedproperty(lambda s: s.property(implies(a, "|->", b))), "(a |-> b)"),
    (lambda u: u.clockedproperty(lambda s: s.clock(posedge).property(implies(a, "|->", b))), "@(posedge clk) a |-> b"),
    (prop(lambda u: u.clockedproperty(lambda s: s.clock(posedge).property(a)), "and", b), "(@(posedge clk) a) and b"),
    # a property spec and a cycle range
    (lambda u: u.propertyspec(lambda s: s.clock(posedge).disable(rst).property(implies(a, "|=>", b))),
     "@(posedge clk) disable iff (rst) a |=> b"),
    (lambda u: u.propertyspec(lambda s: s.disable(rst).property(a)), "disable iff (rst) a"),
    (lambda u: u.propertyspec(lambda s: s.property(a)), "a"), (cycles(one, dollar), "1:$"),
]:
    assert text(spec) == written, (written, text(spec))

<a id="SVA-08"></a>
## SVA-08 · Evaluation refuses each of its kinds by name, as a root and as a call's argument

In [ ]:
# Evaluation (Q11): SystemVerilog's evaluator refuses each of its kinds by name, as a root and where a call's argument
# holds one; what it evaluated before, an expression among them, it still evaluates.
SAMPLES = {
    "delaysequence": delay(a, (one, b)), "repetitionsequence": repeat(a, "*", two), "binarysequence": seq(a, "and", b),
    "parenthesizedsequence": lambda u: u.parenthesizedsequence(lambda s: s.sequence(a)),
    "firstmatchsequence": lambda u: u.firstmatchsequence(lambda s: s.sequence(a)),
    "clockedsequence": lambda u: u.clockedsequence(lambda s: s.clock(posedge).sequence(a)),
    "implicationproperty": implies(a, "|->", b), "binaryproperty": prop(a, "and", b), "unaryproperty": unary("not", a),
    "strengthproperty": lambda u: u.strengthproperty(lambda s: s.keyword("weak").sequence(a)),
    "abortproperty": lambda u: u.abortproperty(lambda s: s.keyword("accept_on").condition(rst).operand(a)),
    "conditionalproperty": lambda u: u.conditionalproperty(lambda s: s.condition(en).consequence(a)),
    "caseproperty": lambda u: u.caseproperty(lambda s: s.expression(n).items([lambda t: t.body(a)])),
    "parenthesizedproperty": lambda u: u.parenthesizedproperty(lambda s: s.property(a)),
    "clockedproperty": lambda u: u.clockedproperty(lambda s: s.clock(posedge).property(a)),
    "propertyspec": lambda u: u.propertyspec(lambda s: s.property(a)), "cyclerange": cycles(one, two),
}
assert list(SAMPLES) == list(SVA.TAGS)
values = {"a": 1, "b": 0, "c": 1, "d": 0, "clk": 1, "rst": 0, "en": 1, "n": 2, "x": 3}
for tag, spec in SAMPLES.items():
    with raises(NotImplementedError, match=f"SystemVerilog Assertions {tag} is not evaluated"):
        SVV.OfAny(held(spec), values)
with raises(NotImplementedError, match="SystemVerilog Assertions delaysequence is not evaluated"):
    SVV.OfAny(held(SV.call("f", SAMPLES["delaysequence"]), "Expressions"), SVV.Scope(values, functions={"f": lambda value: value}))
assert str(SVV.OfAny(held(SV.binary("+", a, n)), values)) == str(SVV.OfAny(held(SV.binary("+", a, n), "Expressions"), values)) == "32'sb00000000000000000000000000000011"